In [1]:

import csv
import json
import re
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from pathlib import Path

OUT = Path("two_sources_results")
OUT.mkdir(exist_ok=True)

ARXIV_IDS = ["2502.05467", "2506.21818", "2501.17539"]

HEADERS = {
    "User-Agent": "ResearchCourseProject/1.0 (educational metadata analysis)",
    "Accept": "application/json, application/atom+xml",
}

def download(url, retries=3):
    for attempt in range(retries):
        try:
            req = urllib.request.Request(url, headers=HEADERS)
            with urllib.request.urlopen(req, timeout=40) as response:
                return response.read()
        except urllib.error.HTTPError as e:
            if e.code == 404:
                return None
            if e.code in (429, 500, 502, 503, 504) and attempt < retries - 1:
                wait = 10 * (attempt + 1)
                print(f"HTTP {e.code}; повтор через {wait} сек.")
                time.sleep(wait)
                continue
            raise

def write_csv(filename, rows, fields):
    with open(OUT / filename, "w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

# ИСТОЧНИК 1: arXiv API
params = urllib.parse.urlencode({"id_list": ",".join(ARXIV_IDS)})
url = "https://export.arxiv.org/api/query?" + params
xml_bytes = download(url)

if not xml_bytes:
    raise RuntimeError("arXiv не вернул данные.")

root = ET.fromstring(xml_bytes)
ns = {"a": "http://www.w3.org/2005/Atom"}
arxiv_rows = []

for entry in root.findall("a:entry", ns):
    article_url = entry.findtext("a:id", default="", namespaces=ns)
    match = re.search(r"/abs/([^/?#]+)", article_url)
    if not match:
        continue
    article_id = re.sub(r"v\d+$", "", match.group(1))
    if article_id not in ARXIV_IDS:
        continue
    arxiv_rows.append({
        "arxiv_id": article_id,
        "arxiv_title": " ".join(
            entry.findtext("a:title", default="", namespaces=ns).split()
        ),
        "arxiv_abstract": " ".join(
            entry.findtext("a:summary", default="", namespaces=ns).split()
        ),
        "arxiv_published": entry.findtext(
            "a:published", default="", namespaces=ns
        ),
    })

if not arxiv_rows:
    raise RuntimeError("arXiv вернул ответ без нужных публикаций.")

write_csv("01_arxiv.csv", arxiv_rows,
          ["arxiv_id", "arxiv_title", "arxiv_abstract", "arxiv_published"])

print("Источник 1 — arXiv:", len(arxiv_rows), "записей")

# ИСТОЧНИК 2: Semantic Scholar API
s2_rows = []
s2_unavailable = []

for i, row in enumerate(arxiv_rows):
    article_id = row["arxiv_id"]
    api_id = urllib.parse.quote("ARXIV:" + article_id, safe="")
    api_url = (
        "https://api.semanticscholar.org/graph/v1/paper/"
        + api_id
        + "?fields=title,citationCount,year,externalIds"
    )
    try:
        response = download(api_url)
        if response is None:
            print("Не найдено в Semantic Scholar:", article_id)
            s2_unavailable.append(article_id)
        else:
            paper = json.loads(response)
            external = paper.get("externalIds") or {}
            returned_id = re.sub(r"v\d+$", "", str(external.get("ArXiv") or ""))
            if returned_id != article_id:
                print("Несовпадение arXiv ID:", article_id)
                s2_unavailable.append(article_id)
            else:
                s2_rows.append({
                    "arxiv_id": returned_id,
                    "s2_paper_id": paper.get("paperId"),
                    "s2_title": paper.get("title"),
                    "s2_year": paper.get("year"),
                    "citation_count": paper.get("citationCount"),
                })
    except Exception as e:
        print("Ошибка Semantic Scholar для", article_id, ":", str(e))
        s2_unavailable.append(article_id)
    if i < len(arxiv_rows) - 1:
        time.sleep(3)

write_csv("02_semantic_scholar.csv", s2_rows,
          ["arxiv_id", "s2_paper_id", "s2_title",
           "s2_year", "citation_count"])

print("Источник 2 — Semantic Scholar:", len(s2_rows), "записей")

# СОЕДИНЕНИЕ: LEFT JOIN по arxiv_id
s2_by_id = {r["arxiv_id"]: r for r in s2_rows}
joined = []

for row in arxiv_rows:
    other = s2_by_id.get(row["arxiv_id"])
    joined.append({
        **row,
        "matched": int(other is not None),
        "s2_paper_id": other["s2_paper_id"] if other else None,
        "s2_title": other["s2_title"] if other else None,
        "s2_year": other["s2_year"] if other else None,
        "citation_count": other["citation_count"] if other else None,
    })

write_csv("03_joined.csv", joined,
          ["arxiv_id", "arxiv_title", "arxiv_abstract",
           "arxiv_published", "matched", "s2_paper_id",
           "s2_title", "s2_year", "citation_count"])

summary = {
    "arxiv_rows": len(arxiv_rows),
    "semantic_scholar_rows": len(s2_rows),
    "joined_rows": len(joined),
    "matched_rows": sum(r["matched"] for r in joined),
    "unmatched_rows": sum(not r["matched"] for r in joined),
    "join_type": "LEFT JOIN",
    "join_key": "arxiv_id",
    "note": "Live API demonstration; not a reprocessing of the 140576-row dataset.",
}

(OUT / "04_summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("\nГОТОВО!")
print(json.dumps(summary, ensure_ascii=False, indent=2))

# Скачать все файлы одним архивом
import shutil
shutil.make_archive("two_sources_results", "zip", OUT)

from google.colab import files
files.download("two_sources_results.zip")


Источник 1 — arXiv: 3 записей
HTTP 429; повтор через 10 сек.
HTTP 429; повтор через 20 сек.
Ошибка Semantic Scholar для 2501.17539 : HTTP Error 429: 
HTTP 429; повтор через 10 сек.
HTTP 429; повтор через 20 сек.
Ошибка Semantic Scholar для 2506.21818 : HTTP Error 429: 
Источник 2 — Semantic Scholar: 1 записей

ГОТОВО!
{
  "arxiv_rows": 3,
  "semantic_scholar_rows": 1,
  "joined_rows": 3,
  "matched_rows": 1,
  "unmatched_rows": 2,
  "join_type": "LEFT JOIN",
  "join_key": "arxiv_id",
  "note": "Live API demonstration; not a reprocessing of the 140576-row dataset."
}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>